In [1]:
import pandas as pd
import optuna
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from imblearn.over_sampling import SMOTE


In [2]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced Dataset.csv')

In [3]:
# Drop diseases with less than 1000 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 1000].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 37
Number of rows left: 44748


In [8]:
# Optuna optimization function
def objective(trial):
    # Define the hyperparameters to tune
    n_estimators = trial.suggest_int('n_estimators', 50, 150)
    max_depth = trial.suggest_int('max_depth', 10, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

    # Create RandomForestClassifier with hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )

    # Perform cross-validation (using 5-fold by default)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    accuracy = score.mean()

    # Log the hyperparameters and accuracy for this trial
    print(f"Trial {trial.number}: n_estimators={n_estimators}, max_depth={max_depth}, "
          f"min_samples_split={min_samples_split}, min_samples_leaf={min_samples_leaf}, "
          f"max_features={max_features}, Accuracy={accuracy:.4f}")

    return accuracy

In [9]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="randomforest_diseases_symptoms_dropextremelymore1000_study", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/randomforest.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=20)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-22 01:55:01,817] A new study created in RDB with name: randomforest_diseases_symptoms_dropextremelymore1000_study
[I 2025-04-22 01:55:15,851] Trial 0 finished with value: 0.5096344985575223 and parameters: {'n_estimators': 88, 'max_depth': 34, 'min_samples_split': 14, 'min_samples_leaf': 8, 'max_features': 'log2'}. Best is trial 0 with value: 0.5096344985575223.


Trial 0: n_estimators=88, max_depth=34, min_samples_split=14, min_samples_leaf=8, max_features=log2, Accuracy=0.5096


[I 2025-04-22 01:56:16,259] Trial 1 finished with value: 0.49953581642274125 and parameters: {'n_estimators': 138, 'max_depth': 48, 'min_samples_split': 12, 'min_samples_leaf': 11, 'max_features': None}. Best is trial 0 with value: 0.5096344985575223.


Trial 1: n_estimators=138, max_depth=48, min_samples_split=12, min_samples_leaf=11, max_features=None, Accuracy=0.4995


[I 2025-04-22 01:56:26,738] Trial 2 finished with value: 0.5080380159790622 and parameters: {'n_estimators': 63, 'max_depth': 34, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.5096344985575223.


...

Trial 17: n_estimators=74, max_depth=11, min_samples_split=20, min_samples_leaf=8, max_features=log2, Accuracy=0.4834


[I 2025-04-22 02:01:17,271] Trial 18 finished with value: 0.509510733883858 and parameters: {'n_estimators': 99, 'max_depth': 37, 'min_samples_split': 9, 'min_samples_leaf': 18, 'max_features': 'sqrt'}. Best is trial 8 with value: 0.5104513164561305.


Trial 18: n_estimators=99, max_depth=37, min_samples_split=9, min_samples_leaf=18, max_features=sqrt, Accuracy=0.5095


[I 2025-04-22 02:02:05,373] Trial 19 finished with value: 0.49765471101129044 and parameters: {'n_estimators': 120, 'max_depth': 45, 'min_samples_split': 4, 'min_samples_leaf': 13, 'max_features': None}. Best is trial 8 with value: 0.5104513164561305.


Trial 19: n_estimators=120, max_depth=45, min_samples_split=4, min_samples_leaf=13, max_features=None, Accuracy=0.4977

Best Trial:
FrozenTrial(number=8, state=TrialState.COMPLETE, values=[0.5104513164561305], datetime_start=datetime.datetime(2025, 4, 22, 1, 58, 25, 961602), datetime_complete=datetime.datetime(2025, 4, 22, 1, 58, 46, 687112), params={'n_estimators': 131, 'max_depth': 44, 'min_samples_split': 8, 'min_samples_leaf': 11, 'max_features': 'sqrt'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=150, log=False, low=50, step=1), 'max_depth': IntDistribution(high=50, log=False, low=10, step=1), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=20, log=False, low=1, step=1), 'max_features': CategoricalDistribution(choices=('sqrt', 'log2', None))}, trial_id=126, value=None)
Best Hyperparameters:
{'n_estimators': 131, 'max_depth': 44, 'min_samples_split': 8,